# 🚴 Cálculo de Potencia Crítica (CP) y Capacidad de Trabajo Glucolítico (W')

---

## ¿Qué es la Potencia Crítica?

La **Potencia Crítica (CP)** representa la máxima potencia que un atleta puede sostener teóricamente por un tiempo indefinido sin fatiga progresiva. Es un parámetro clave del perfil de rendimiento aeróbico.

La **Capacidad de Trabajo Glucolítico (W')** representa una "reserva" finita de energía que se puede utilizar por encima de CP.

### Modelo Hiperbólico:

$$P = \frac{W'}{t} + CP$$

Donde:
- **P** = Potencia (Watts)
- **t** = Tiempo (segundos)
- **CP** = Potencia Crítica (Watts)
- **W'** = Capacidad de trabajo glucolítico de alta intensidad (Joules)

---

### 📋 Instrucciones:
1. Ejecuta cada celda en orden (▶ o `Shift + Enter`)
2. En la **Celda 3**, ingresa tus datos de potencia máxima para cada duración
3. Elige si quieres comparar el modelo con 2 vs 3 puntos
4. Observa los resultados, el gráfico y las zonas de entrenamiento
5. Exporta tus resultados al final

> ⚠️ Los valores de potencia deben ser esfuerzos **máximos sostenidos** para esa duración (test de campo o laboratorio)

In [ ]:
# ============================================================
# CELDA 1 — Instalación e importación de librerías
# ============================================================
# Ejecuta esta celda primero. Solo necesitas hacerlo una vez.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import curve_fit
from scipy.stats import pearsonr
import pandas as pd
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas correctamente')
print('➡️  Continúa con la Celda 2')

In [ ]:
# ============================================================
# CELDA 2 — Funciones del modelo
# ============================================================

def modelo_cp(t, CP, W_prima):
    """Modelo hiperbólico: P = W'/t + CP"""
    return W_prima / t + CP

def calcular_cp(tiempos_seg, potencias):
    """
    Ajusta el modelo CP usando mínimos cuadrados no lineales.
    Retorna CP, W' y métricas de ajuste.
    """
    t = np.array(tiempos_seg, dtype=float)
    P = np.array(potencias, dtype=float)

    # Estimación inicial: CP ~ potencia más baja, W' estimado
    CP_init = min(P) * 0.85
    W_init = (P[0] - CP_init) * t[0]

    try:
        params, _ = curve_fit(
            modelo_cp, t, P,
            p0=[CP_init, W_init],
            bounds=([0, 0], [np.inf, np.inf]),
            maxfev=10000
        )
        CP, W_prima = params

        # R² del ajuste
        P_pred = modelo_cp(t, CP, W_prima)
        ss_res = np.sum((P - P_pred) ** 2)
        ss_tot = np.sum((P - np.mean(P)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0

        return CP, W_prima, r2
    except Exception as e:
        print(f'Error en ajuste: {e}')
        return None, None, None

def zonas_entrenamiento(CP):
    """Devuelve zonas de entrenamiento basadas en CP."""
    zonas = {
        'Z1 — Recuperación activa': (0, round(CP * 0.55)),
        'Z2 — Resistencia aeróbica': (round(CP * 0.55), round(CP * 0.75)),
        'Z3 — Tempo / Umbral':       (round(CP * 0.75), round(CP * 0.90)),
        'Z4 — Sub-CP / Umbral alto': (round(CP * 0.90), round(CP * 1.00)),
        'Z5 — Sobre CP (W\' domain)':(round(CP * 1.00), round(CP * 1.20)),
        'Z6 — Neuro-muscular >120%CP':(round(CP * 1.20), '∞'),
    }
    return zonas

print('✅ Funciones definidas')
print('➡️  Continúa con la Celda 3 para ingresar tus datos')

In [ ]:
# ============================================================
# CELDA 3 — 📝 INGRESO DE DATOS (¡EDITA AQUÍ!)
# ============================================================
#
# Instrucciones:
#   - Ingresa la potencia MÁXIMA SOSTENIDA para cada duración
#   - Las duraciones deben estar entre 1 y 20 minutos
#   - Usa al menos 2 puntos (recomendado: 3 o más)
#   - Para desactivar un punto, pon None en la potencia
#
#   Rango corto  → 1 a 10 minutos  (ej: 1, 2, 3, 5 min)
#   Rango largo  → 10 a 20 minutos (ej: 12, 16, 20 min)
# ============================================================

# 👇 MODIFICA ESTOS VALORES CON TUS DATOS REALES
datos_atleta = {
    'nombre': 'Atleta 1',          # ← Cambia el nombre
    'peso_kg': 70,                  # ← Peso corporal en kg
    'deporte': 'Ciclismo',         # ← Ciclismo / Running / Natación
}

# Duraciones en MINUTOS y Potencias en WATTS
# Formato: (duración_min, potencia_W)
# ↓↓↓ EDITA ESTOS PARES DE VALORES ↓↓↓

mis_datos = [
    #  (min,  Watts)
    (   1,    480  ),   # Esfuerzo de 1 minuto
    (   3,    390  ),   # Esfuerzo de 3 minutos
    (   5,    340  ),   # Esfuerzo de 5 minutos
    (  12,    290  ),   # Esfuerzo de 12 minutos
    (  20,    265  ),   # Esfuerzo de 20 minutos
    # Agrega o elimina filas según necesites
    # ( min,  None ),   # ← Desactiva un punto con None
]

# ============================================================
# Procesamiento automático (no necesitas editar esto)
# ============================================================
datos_validos = [(t, p) for t, p in mis_datos if p is not None and t >= 1 and t <= 20]
datos_validos.sort(key=lambda x: x[0])

tiempos_min = [d[0] for d in datos_validos]
tiempos_seg = [d[0] * 60 for d in datos_validos]
potencias   = [d[1] for d in datos_validos]

print(f'\n👤 Atleta: {datos_atleta["nombre"]} | Peso: {datos_atleta["peso_kg"]} kg | Deporte: {datos_atleta["deporte"]}')
print(f'\n📊 Puntos ingresados ({len(datos_validos)} puntos):')
print('-' * 35)
print(f'{"Duración":>12}  {"Potencia":>10}  {"W/kg":>8}')
print('-' * 35)
for t, p in zip(tiempos_min, potencias):
    wpkg = p / datos_atleta['peso_kg']
    print(f'{str(t)+" min":>12}  {str(p)+" W":>10}  {wpkg:>7.2f}')
print('-' * 35)

if len(datos_validos) < 2:
    print('\n⚠️  Necesitas al menos 2 puntos para calcular CP')
else:
    print(f'\n✅ Datos válidos. Continúa con la Celda 4')

In [ ]:
# ============================================================
# CELDA 4 — Cálculo de CP con todos los puntos
# ============================================================

print('=' * 50)
print('  RESULTADOS — MODELO COMPLETO (todos los puntos)')
print('=' * 50)

CP_total, W_total, r2_total = calcular_cp(tiempos_seg, potencias)

if CP_total:
    W_prima_kJ = W_total / 1000
    CP_wkg = CP_total / datos_atleta['peso_kg']

    print(f'\n  Potencia Crítica (CP):  {CP_total:.1f} W  ({CP_wkg:.2f} W/kg)')
    print(f'  W\' (capacidad glucolítica): {W_prima_kJ:.1f} kJ  ({W_total:.0f} J)')
    print(f'  Ajuste del modelo (R²):   {r2_total:.4f}')
    print(f'  Puntos usados:            {len(datos_validos)}')

    if r2_total >= 0.99:
        calidad = '🟢 Excelente'
    elif r2_total >= 0.97:
        calidad = '🟡 Bueno'
    else:
        calidad = '🔴 Verificar datos'
    print(f'  Calidad del ajuste:       {calidad}')
    print('\n' + '=' * 50)

    # Guardar resultados para celdas siguientes
    resultados_globales = {
        'CP': CP_total, 'W_prima': W_total, 'R2': r2_total,
        'n_puntos': len(datos_validos)
    }
else:
    print('\n❌ No se pudo ajustar el modelo. Revisa tus datos.')

In [ ]:
# ============================================================
# CELDA 5 — Comparación: 2 puntos vs 3 puntos
# ============================================================
#
# Esta celda muestra cómo cambia la estimación de CP
# según cuántos y cuáles puntos se usan en el modelo.
# ¡Observa las diferencias!

print('=' * 60)
print('  COMPARACIÓN: 2 puntos vs 3 puntos vs modelo completo')
print('=' * 60)

n = len(datos_validos)
comparaciones = []

if n >= 2:
    # Con los 2 extremos (punto más corto y más largo)
    idx_2 = [0, n-1]
    t2 = [tiempos_seg[i] for i in idx_2]
    p2 = [potencias[i] for i in idx_2]
    CP2, W2, r2_2 = calcular_cp(t2, p2)
    label2 = f'{tiempos_min[0]}min + {tiempos_min[-1]}min'
    comparaciones.append(('2 puntos (extremos)', label2, CP2, W2, r2_2))

if n >= 3:
    # Con el punto corto, uno del medio y el más largo
    idx_3 = [0, n//2, n-1]
    t3 = [tiempos_seg[i] for i in idx_3]
    p3 = [potencias[i] for i in idx_3]
    CP3, W3, r2_3 = calcular_cp(t3, p3)
    label3 = f'{tiempos_min[0]}min + {tiempos_min[n//2]}min + {tiempos_min[-1]}min'
    comparaciones.append(('3 puntos (distribuidos)', label3, CP3, W3, r2_3))

# Modelo completo
comparaciones.append((
    f'Modelo completo ({n} puntos)',
    'Todos los puntos',
    CP_total, W_total, r2_total
))

# Mostrar tabla comparativa
print(f'\n{"Modelo":<30} {"Puntos usados":<28} {"CP (W)":>7} {"W\' (kJ)":>9} {"R²":>7}')
print('-' * 85)
for nombre, puntos_label, cp, wp, r2 in comparaciones:
    if cp:
        print(f'{nombre:<30} {puntos_label:<28} {cp:>7.1f} {wp/1000:>9.1f} {r2:>7.4f}')

print('-' * 85)
print('\n💡 Reflexión:')
print('   ¿Cuánto varía la CP entre modelos? ¿Qué implicancias tiene')
print('   para el entrenamiento usar solo 2 puntos vs más puntos?')

In [ ]:
# ============================================================
# CELDA 6 — Gráfico: Curva Potencia-Duración
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Curva Potencia-Duración — {datos_atleta["nombre"]}', fontsize=14, fontweight='bold')

# Rango de tiempo para la curva teórica
t_plot_seg = np.linspace(60, 1200, 500)  # 1 a 20 min
t_plot_min = t_plot_seg / 60

colores_mod = ['#E74C3C', '#F39C12', '#2ECC71']

# ---- PANEL IZQUIERDO: Escala lineal ----
ax1 = axes[0]
ax1.set_title('Escala lineal', fontsize=11)

for i, (nombre, puntos_label, cp, wp, r2) in enumerate(comparaciones):
    if cp:
        P_curva = modelo_cp(t_plot_seg, cp, wp)
        ax1.plot(t_plot_min, P_curva,
                 color=colores_mod[i % len(colores_mod)],
                 linewidth=2.5 if i == len(comparaciones)-1 else 1.5,
                 linestyle='-' if i == len(comparaciones)-1 else '--',
                 label=f'{nombre} | CP={cp:.0f}W R²={r2:.4f}',
                 alpha=0.9)

# CP como línea horizontal
ax1.axhline(y=CP_total, color='#2ECC71', linewidth=1.2, linestyle=':', alpha=0.7)
ax1.text(19.5, CP_total + 5, f'CP={CP_total:.0f}W', ha='right', fontsize=9, color='#27AE60')

# Puntos observados
ax1.scatter(tiempos_min, potencias, zorder=5, s=80, color='#2C3E50',
            edgecolors='white', linewidth=1.5, label='Datos observados')

ax1.set_xlabel('Duración (minutos)', fontsize=11)
ax1.set_ylabel('Potencia (W)', fontsize=11)
ax1.legend(fontsize=8, loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0.5, 21)

# ---- PANEL DERECHO: Escala logarítmica ----
ax2 = axes[1]
ax2.set_title('Escala semi-logarítmica (común en literatura)', fontsize=11)

for i, (nombre, puntos_label, cp, wp, r2) in enumerate(comparaciones):
    if cp:
        P_curva = modelo_cp(t_plot_seg, cp, wp)
        ax2.plot(t_plot_min, P_curva,
                 color=colores_mod[i % len(colores_mod)],
                 linewidth=2.5 if i == len(comparaciones)-1 else 1.5,
                 linestyle='-' if i == len(comparaciones)-1 else '--',
                 alpha=0.9)

ax2.axhline(y=CP_total, color='#2ECC71', linewidth=1.2, linestyle=':', alpha=0.7)
ax2.text(19, CP_total + 5, f'CP={CP_total:.0f}W', ha='right', fontsize=9, color='#27AE60')
ax2.scatter(tiempos_min, potencias, zorder=5, s=80, color='#2C3E50',
            edgecolors='white', linewidth=1.5)
ax2.set_xscale('log')
ax2.set_xlabel('Duración (minutos, escala log)', fontsize=11)
ax2.set_ylabel('Potencia (W)', fontsize=11)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('curva_pd.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como curva_pd.png')

In [ ]:
# ============================================================
# CELDA 7 — Zonas de Entrenamiento basadas en CP
# ============================================================

print('=' * 55)
print(f'  ZONAS DE ENTRENAMIENTO — {datos_atleta["nombre"]}')
print(f'  Basadas en CP = {CP_total:.1f} W  ({CP_wkg:.2f} W/kg)')
print('=' * 55)

zonas = zonas_entrenamiento(CP_total)

colores_zonas = ['#85C1E9', '#82E0AA', '#F9E79F', '#F0B27A', '#E59866', '#C0392B']

print(f'\n{"Zona":<32} {"Rango (W)":>14}  {"Rango (W/kg)":>14}')
print('-' * 63)
for (nombre_zona, (p_min, p_max)) in zonas.items():
    if p_max == '∞':
        rango_w   = f'> {p_min} W'
        rango_wkg = f'> {p_min/datos_atleta["peso_kg"]:.2f}'
    else:
        rango_w   = f'{p_min} – {p_max} W'
        rango_wkg = f'{p_min/datos_atleta["peso_kg"]:.2f} – {p_max/datos_atleta["peso_kg"]:.2f}'
    print(f'{nombre_zona:<32} {rango_w:>14}  {rango_wkg:>14}')
print('-' * 63)

# Gráfico de zonas
fig, ax = plt.subplots(figsize=(12, 3.5))
nombres_zonas = list(zonas.keys())
rangos = list(zonas.values())

x_prev = 0
for i, (nombre_z, (p_min, p_max)) in enumerate(zonas.items()):
    ancho = (p_max if p_max != '∞' else CP_total * 1.4) - p_min
    ax.barh(0, ancho, left=p_min, height=0.6,
            color=colores_zonas[i], edgecolor='white', linewidth=2)
    centro = p_min + ancho / 2
    etiqueta = nombre_z.split('—')[0].strip()
    ax.text(centro, 0, etiqueta, ha='center', va='center', fontsize=8, fontweight='bold')

ax.axvline(x=CP_total, color='black', linewidth=2, linestyle='--')
ax.text(CP_total + 2, 0.35, f'CP\n{CP_total:.0f}W', fontsize=9, fontweight='bold')
ax.set_xlabel('Potencia (W)', fontsize=11)
ax.set_yticks([])
ax.set_title(f'Zonas de Entrenamiento — {datos_atleta["nombre"]}', fontsize=12, fontweight='bold')
ax.set_xlim(0, CP_total * 1.45)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('zonas_entrenamiento.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico de zonas guardado como zonas_entrenamiento.png')

In [ ]:
# ============================================================
# CELDA 8 — Exportar resultados a CSV y Excel
# ============================================================

from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
nombre_archivo = f'CP_{datos_atleta["nombre"].replace(" ","_")}_{timestamp}'

# --- DataFrame 1: Datos ingresados ---
df_datos = pd.DataFrame({
    'Duracion_min': tiempos_min,
    'Duracion_seg': tiempos_seg,
    'Potencia_W': potencias,
    'Potencia_Wkg': [round(p/datos_atleta['peso_kg'], 2) for p in potencias]
})

# --- DataFrame 2: Resultados CP ---
filas_resultados = []
for nombre_mod, puntos_label, cp, wp, r2 in comparaciones:
    if cp:
        filas_resultados.append({
            'Modelo': nombre_mod,
            'Puntos_usados': puntos_label,
            'CP_W': round(cp, 1),
            'CP_Wkg': round(cp/datos_atleta['peso_kg'], 2),
            'W_prima_J': round(wp, 0),
            'W_prima_kJ': round(wp/1000, 1),
            'R2': round(r2, 4)
        })
df_resultados = pd.DataFrame(filas_resultados)

# --- DataFrame 3: Zonas ---
filas_zonas = []
for nombre_z, (p_min, p_max) in zonas_entrenamiento(CP_total).items():
    filas_zonas.append({
        'Zona': nombre_z,
        'Potencia_min_W': p_min,
        'Potencia_max_W': p_max if p_max != '∞' else '>'+str(p_min),
    })
df_zonas = pd.DataFrame(filas_zonas)

# --- Guardar CSV ---
df_datos.to_csv(f'{nombre_archivo}_datos.csv', index=False)
df_resultados.to_csv(f'{nombre_archivo}_resultados.csv', index=False)
df_zonas.to_csv(f'{nombre_archivo}_zonas.csv', index=False)

# --- Guardar Excel (multi-hoja) ---
with pd.ExcelWriter(f'{nombre_archivo}.xlsx', engine='openpyxl') as writer:
    df_datos.to_excel(writer, sheet_name='Datos ingresados', index=False)
    df_resultados.to_excel(writer, sheet_name='Resultados CP', index=False)
    df_zonas.to_excel(writer, sheet_name='Zonas entrenamiento', index=False)

print('=' * 50)
print('  ✅ EXPORTACIÓN COMPLETADA')
print('=' * 50)
print(f'\n  📄 {nombre_archivo}_datos.csv')
print(f'  📄 {nombre_archivo}_resultados.csv')
print(f'  📄 {nombre_archivo}_zonas.csv')
print(f'  📊 {nombre_archivo}.xlsx  (3 hojas)')
print(f'  🖼️  curva_pd.png')
print(f'  🖼️  zonas_entrenamiento.png')
print(f'\n  📥 Para descargar: panel izquierdo → ícono 📁 → selecciona el archivo')

print('\n' + '=' * 50)
print('  RESUMEN FINAL')
print('=' * 50)
print(f'  Atleta:  {datos_atleta["nombre"]} | {datos_atleta["peso_kg"]} kg')
print(f'  CP:      {CP_total:.1f} W  ({CP_wkg:.2f} W/kg)')
print(f'  W\':      {W_total/1000:.1f} kJ')
print(f'  R²:      {r2_total:.4f}')
print(f'  Puntos:  {len(datos_validos)}')
print('=' * 50)

---
## 📚 Preguntas de reflexión para el informe

1. **¿Cómo afecta la elección de puntos?** Compara el CP obtenido con 2 puntos vs el modelo completo. ¿Cuál es la diferencia en watts? ¿Qué implicancias tiene esto en la práctica?

2. **Calidad del ajuste:** ¿Qué tan bien se ajusta el modelo hiperbólico a tus datos (R²)? ¿Qué podría explicar un ajuste bajo?

3. **Interpretación del W':** Si un atleta tiene W' = 20 kJ y supera la CP en 50 W, ¿cuántos segundos puede sostener ese esfuerzo?

4. **Zonas de entrenamiento:** ¿En qué zona corresponde un esfuerzo de 30 minutos al ritmo de competencia? ¿Y un sprint de 15 segundos?

5. **Limitaciones del modelo:** El modelo hiperbólico tiene supuestos que no siempre se cumplen. ¿Cuáles conoces? ¿En qué contextos puede fallar?

---
*Notebook desarrollado para curso de Fisiología del Ejercicio — Pregrado*